# Assignment: End-to-End Classification Workflow

Predicting Earthquake Damage with Classification Models

## Introduction

In this assignment, you will perform an **end-to-end classification workflow** for **Kavrepalanchok district** 🇳🇵. Unlike the previous lessons — which walked you through each step with guided examples — this assignment places you in the driver's seat: you are responsible for the entire pipeline, from querying the database to evaluating your model.

**Your Goal:** Predict building damage in Kavrepalanchok district using the skills you learned in Lessons 1-4.

Kavrepalanchok is a district east of Kathmandu that suffered significant damage in the 2015 Gorkha earthquake. It has **more buildings than Gorkha district** — making it a larger and somewhat different prediction challenge.

Before diving into code, we'll consolidate the key conceptual foundations from Lessons 1-4 that underpin every step of this assignment. Understanding *why* each step exists helps you make informed decisions — not just follow recipes.

**Assignment structure:**
1. **Data Acquisition** — SQL query to load Kavrepalanchok data
2. **Preparation and Wrangling** — clean, split, encode
3. **Modeling and Evaluation** — train a Decision Tree, compute metrics
4. **Communication and Interpretation** — feature importance, visualization
5. **Summary and Discussion** — reflect on findings and ethical implications

By the end of this assignment, you will have produced a complete working classification pipeline you can adapt to any district or dataset in the P4 project framework.


## Part 1: What Are We Predicting? Features, Target, and Classification

### Classification vs Regression

There are two main types of supervised learning:

**Regression** predicts a **continuous number:**
- Example: house price prediction → outputs "\$250,000"
- Example: earthquake damage score → outputs "42 out of 100"
- The output can take any value along a continuous scale

**Classification** predicts a **discrete category:**
- Example: spam detection → outputs "spam" or "not spam"
- Example: building damage severity → outputs "severe" or "not severe"
- The output belongs to one of a fixed set of classes

In this assignment, we do **binary classification** — exactly two possible outcomes: `severe_damage = 1` (Grade 4 or 5) or `severe_damage = 0` (Grade 1, 2, or 3).

> 💡 **Why not regression?** We could predict the raw damage grade (1-5) as a continuous number. But the *policy decision* — evacuate or don't evacuate — is binary. A score of 3.7 doesn't tell a responder whether to evacuate. "Severe: yes/no" does. We design the ML task to match the real-world decision.


### Features vs Target — A Clear Distinction

**Features (X)** are the **inputs** — building characteristics we know *before* the earthquake:

| Features (what we know before) |
|---------------------------------|
| `foundation_type`               |
| `height_ft_pre_eq`              |
| `age_building`                  |
| `roof_type`                     |
| `land_surface_condition`        |
| `caste_household` (from demographics) |
| …                               |

**Target (y)** is what we're **predicting** — the outcome:

| Target (what we predict)             |
|--------------------------------------|
| `severe_damage = 1` (Grade 4 or 5)   |
| `severe_damage = 0` (Grade 1, 2, or 3) |

The model learns the mapping: *given features X, predict target y*.

Every row in the training dataset has **both** features AND a known target — that's what makes it *supervised* learning. In deployment (predicting damage for a new earthquake), only features are available; the model outputs the predicted target.

> ⚠️ **The leakage rule:** if a column is derived from or directly correlated with the target — like `damage_grade` (the raw form of `severe_damage`) or any post-earthquake measurement — it **cannot be used as a feature**. The model would be using future information to predict the past. This appears to work perfectly during training but fails completely in deployment.


## Part 2: Decision Tree Concepts — Splits, Impurity, and Depth

### How a Decision Tree Makes Decisions

A decision tree makes predictions by asking a series of **binary (yes/no) questions** about feature values, routing each building down a path to a final prediction leaf.

```
                Is foundation_type ≤ 2?          ← Internal node: binary question
                       /       \
                    YES          NO
                     /            \
           Is height > 15?    Predict: Severe     ← Leaf: final prediction
               /        \
             YES         NO
              /            \
        Predict:        Predict:
       Not Severe         Severe
```

Key terminology:
- **Internal node**: a binary question about one feature (`feature ≤ threshold`)
- **Leaf node**: a final prediction (the majority class among training examples that reached this leaf)
- **Tree depth**: the maximum number of questions asked before predicting
- **Root node**: the very first question — the most important single split in the tree

**How the tree chooses which question to ask:** at every node, the algorithm evaluates every possible feature and every possible threshold, then picks the one that most reduces **Gini impurity** — a measure of how mixed the two classes are in each resulting group.


### Gini Impurity: Measuring "Mixedness"

The algorithm doesn't guess questions randomly — it chooses the split that most reduces **Gini impurity**.

**The formula:**
```
Gini = 2 × p × (1 - p)
```
where `p` is the fraction of the majority class.

| Node composition | p | Gini | Interpretation |
|-----------------|---|------|----------------|
| 100% severe | 1.0 | 0.0 | Perfectly pure — confident prediction |
| 80% severe | 0.8 | 0.32 | Mostly one class |
| 50% / 50% | 0.5 | 0.50 | Maximally uncertain |
| 100% not severe | 0.0 | 0.0 | Perfectly pure — confident prediction |

**What the algorithm wants:** low Gini (pure nodes). It evaluates every possible split for every feature and picks the one that reduces the weighted average Gini the most across both child nodes.

> 📌 **Pure leaves = confident predictions; mixed leaves = uncertain guesses.** A tree with max_depth=1 makes one split — one binary question — creating two child nodes. If that split produces very pure children (Gini ≈ 0), the tree is immediately useful. If the children are still 50/50, the tree has learned nothing.


### Shallow vs Deep: The Overfitting Spectrum

Tree depth controls the fundamental bias-variance tradeoff:

| Depth | Behavior | Training Accuracy | Validation Accuracy | Diagnosis |
|-------|----------|-------------------|--------------------|-----------|
| 1 | One question only | 60% | 59% | **Underfitting** — too simple to capture patterns |
| 6 | Balanced splits | 74% | 72% | **Good generalization** — the sweet spot |
| 20 | Memorizes training data | 99% | 68% | **Overfitting** — learned training noise |

**The gap at depth 20 is the key signal:** training accuracy 99%, validation 68% — a 31% gap. The model memorized every quirk of the training buildings and fails to generalize to new ones.

**Why `max_depth=10` in this assignment?**

Without a depth limit, a tree grows until every leaf contains exactly one training example — achieving 100% training accuracy through pure memorization. In Lesson 3, we ran validation curve experiments on Gorkha district and found `max_depth` in the range 6–10 gave the best generalization. `max_depth=10` is a principled starting point; in production, you would tune it precisely via cross-validation.

> 🧠 **The universal principle:** increasing model complexity always reduces training error; it may or may not reduce test error. The point where test error starts increasing (while training error keeps falling) is the overfitting threshold. `max_depth` controls this threshold for Decision Trees.


### Overfitting, Generalization, and the Train-Test Split

**Overfitting**: the model learns the noise and quirks of the training data, performing poorly on unseen data.

**Generalization**: the model's ability to perform well on data it has never seen before — the actual goal.

**The Train-Test Split for Kavrepalanchok:**

```
Full Dataset (~82,684 buildings in Kavrepalanchok)
├── Training Set (80% = ~66,147 buildings) ── model LEARNS from this
└── Test Set    (20% = ~16,537 buildings) ── model EVALUATED here (never seen during training)
```

**The exam analogy:** a student who memorizes past exam questions scores 100% on those exact questions. But if the final exam has *different* questions, they might fail. Training accuracy = past exam scores. Test accuracy = the real final exam.

**Why we must never use test data during training or hyperparameter tuning:**

Every time you look at test set performance to make a modeling decision (choosing max_depth, choosing features, choosing algorithms), you are implicitly fitting to the test set. After many such decisions, your test accuracy is no longer a honest estimate of out-of-sample performance.

> ⚠️ **In Lesson 3, we introduced a validation set precisely for this reason.** In this assignment, max_depth is given (=10), so we use a simple two-way split. But in a full production workflow, you would use a three-way split or cross-validation.


---

## 1. Data Acquisition

Let's begin by importing all the libraries needed for this assignment.

**Code 4.5.0.1**: Import required libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from category_encoders import OneHotEncoder

# Set display options
pd.set_option('display.max_columns', None)

**Code Task 4.5.0.1**: In this assignment, you will analyze a different district from the one we used in Lessons 2-4. Start by running the cell below to count buildings per district — this tells you the district ID for Kavrepalanchok.

> 💡 **Kavrepalanchok context:** this district is east of Kathmandu Valley. It experienced severe shaking in the 2015 Gorkha earthquake despite being farther from the epicenter than Gorkha itself, largely due to local geology (sedimentary basins that amplify seismic waves). The building stock in Kavrepalanchok is similar in composition to Gorkha but with some differences in caste demographics and construction practices.

**Code 4.5.0.1**


In [ ]:
# Count buildings per district using DuckDB
district_counts = duckdb.sql("""
    SELECT district_id, COUNT(DISTINCT building_id) as building_count
    FROM './data/id_map.csv'
    GROUP BY district_id
    ORDER BY district_id
""").fetchdf()

print("Building count by district:")
print(district_counts)

Based on the output above, identify which `district_id` has the most buildings. **That's Kavrepalanchok (district 3)!**

> 📌 The district IDs in this database are: 1 = Sindhupalchok, 2 = Nuwakot, 3 = Kavrepalanchok, 4 = Gorkha, and others. Kavrepalanchok (district 3) is one of the most populous districts affected by the earthquake.

Now assign the Kavrepalanchok district ID to `district_id_assign`. This variable will be used in subsequent SQL queries.

**Code Task 4.5.0.1**


In [ ]:
# Kavrepalanchok is district_id 3
district_id_assign = 3

print(f"Kavrepalanchok district ID: {district_id_assign}")

**Solution 4.5.0.1**

In [ ]:
# Kavrepalanchok is district_id 3
district_id_assign = 3

print(f"Kavrepalanchok district ID: {district_id_assign}")

---

**Code Task 4.5.1.1**: Write a SQL query using DuckDB to load the complete Kavrepalanchok dataset. Your query should:

1. Join `building_structure`, `household_demographics`, `building_damage`, and `id_map` using INNER JOINs
2. Filter to `district_id = district_id_assign`
3. Create the binary `severe_damage` column: `CASE WHEN damage_grade IN ('Grade 4', 'Grade 5') THEN 1 ELSE 0 END`
4. Exclude post-earthquake columns and direct target proxies

The result should be stored in `df_assignment_final`.

> 📌 **This is the same four-table JOIN pattern from Lesson 4** — the core SQL skill of P4. If you need a refresher, review the `id_map` bridge table schema: household_demographics → (household_id) → id_map → (building_id) → building_structure / building_damage.

**Code 4.5.1.1**


In [ ]:
import duckdb

# Connect to database
conn = duckdb.connect('./nepal.sqlite')

# Write your SQL query here - use the district_id_assign variable
query = f"""
    SELECT
        s.*,
        d.damage_grade
    FROM building_structure s
    JOIN building_damage d ON s.building_id = d.building_id
    JOIN id_map i ON s.building_id = i.building_id
    WHERE i.district_id = {district_id_assign}
"""

# Execute query and load into DataFrame
df_assignment_final = conn.execute(query).df()
conn.close()

# Create binary target variable
df_assignment_final = (df_assignment_final
  .assign(severe_damage=df_assignment_final['damage_grade']        # <--- Create 'severe_damage' column from 'damage_grade'
    .str.contains('Grade 4|Grade 5')                              # <--- Check if grade is 4 or 5 (returns True/False)
    .astype(int)))                                                # <--- Convert True→1, False→0

# Drop leaky columns
cols_to_drop = [col for col in df_assignment_final.columns if 'post_eq' in col] + \
               ['building_id', 'damage_grade', 'count_floors_pre_eq']
df_assignment_final = df_assignment_final.drop(columns=cols_to_drop)

print(f"Shape: {df_assignment_final.shape}")
print(df_assignment_final.head())

---

## Part 3: Data Science Workflow — EDA, Encoding, and Feature Engineering

### Exploratory Data Analysis (EDA) — Before Any Modeling

Before training a model, we explore the data to understand what we're working with. EDA catches problems early and informs downstream decisions.

**EDA checklist for this assignment:**

```python
df_assignment_final.shape              # How many rows and columns?
df_assignment_final.isnull().sum()     # Missing values per column
df_assignment_final.dtypes             # Data types — which are categorical?
df_assignment_final['severe_damage'].value_counts(normalize=True)  # Class balance
df_assignment_final.describe()         # Numeric summaries
```

**Why each check matters:**
- `shape`: verify the expected number of Kavrepalanchok buildings (~82,000+)
- `isnull().sum()`: missing values break most ML algorithms without handling
- `dtypes`: categorical columns must be encoded before the model can process them
- `value_counts(normalize=True)`: class balance determines whether accuracy alone is meaningful

### Categorical Encoding — Why ML Needs Numbers

Machine learning algorithms perform mathematical operations (distances, multiplications, gradients). A string like `"Mud mortar-Stone/Brick"` cannot be multiplied or subtracted mathematically.

**One-Hot Encoding (OHE)** converts each categorical value into binary columns:

| `foundation_type` | Mud | Brick | RC | Timber | Other |
|---|---|---|---|---|---|
| Mud mortar-Stone/Brick | 1 | 0 | 0 | 0 | 0 |
| RC engineered | 0 | 0 | 1 | 0 | 0 |
| Bamboo/Timber | 0 | 0 | 0 | 1 | 0 |

Each unique value becomes its own binary column. A building either has (1) or doesn't have (0) that foundation type — no ordinal relationship implied.

**Critical rule:** Fit the encoder on the **training set only**, then transform the test set. This prevents **leakage** — test-set category distribution influencing training encoding. Using `Pipeline` enforces this automatically.

### Why `severe_damage` Is Binary

The original `damage_grade` has 5 categories (Grades 1–5). The binary conversion reflects:

1. **Actionable threshold**: Grades 4–5 indicate structural danger requiring evacuation; Grades 1–3 do not
2. **Clear policy boundary**: maps to "Should this building be evacuated? yes/no"
3. **Aligns with real-world use**: disaster relief teams issue binary occupancy certificates

**What does this imply?** Feature engineering choices should reflect the real-world decision, not mathematical convenience.

### Why `max_depth=10`

From Lesson 3's validation curve experiments on Gorkha data: `max_depth` in the range 6–10 gave the best validation accuracy. We use `max_depth=10` as a principled starting point for Kavrepalanchok. In a full production workflow, you would run a new validation curve for this district.

---

## 2. Preparation and Wrangling

**Code Task 4.5.2.1**: Verify that `'severe_damage'` exists in your DataFrame and contains only 0s and 1s. Store the count of severe damage cases (value = 1) in `severe_damage_count`.

In [ ]:
# Count severe damage cases
severe_damage_count = df_assignment_final['severe_damage'].sum() # <--- 'severe_damage'

print(f"Severe damage cases: {severe_damage_count}")
print(f"Total cases: {len(df_assignment_final)}")
print(f"Percentage: {severe_damage_count / len(df_assignment_final) * 100:.1f}%")

Now split the data into features and target, then apply an 80/20 train-test split.

> 📌 **Why 80/20?** With ~82,000 buildings, an 80/20 split gives ~65,000 for training and ~16,400 for testing — both large enough for reliable estimates. A 70/30 or 75/25 split would also be defensible. The key is that the test set is never touched until final evaluation.

**Code Task 4.5.2.2**: Create feature matrix `X_assign_final` (all columns except `severe_damage`) and target vector `y_assign_final` (the `severe_damage` column). Then apply an 80/20 train-test split with `random_state=42`. Store as `X_train_assign`, `X_test_assign`, `y_train_assign`, `y_test_assign`.


In [ ]:
X_assign_final = df_assignment_final.drop(columns=['severe_damage'])   # <--- 'severe_damage'
y_assign_final = df_assignment_final['severe_damage']                  # <--- 'severe_damage' 

X_train_assign, X_test_assign, y_train_assign, y_test_assign = train_test_split(
    X_assign_final, y_assign_final, test_size=0.2, random_state=42              # <---  X_assign_final, y_assign_final
)

Now encode the categorical features using `OneHotEncoder`. Remember the golden rule: **fit only on training data, then transform both train and test.**

> ⚠️ **`use_cat_names=True`** preserves original category names in the encoded column names — e.g., `foundation_type_RC_engineered` instead of `foundation_type_3`. This makes the encoded feature matrix interpretable: when we look at feature importances, we can directly read "RC foundation matters" instead of guessing what `foundation_type_3` means.

**Code Task 4.5.2.3**: Create a `OneHotEncoder(use_cat_names=True)` instance named `encoder_final`. Use `fit_transform` on `X_train_assign` to get `X_train_assign_enc`. Use `transform` (not `fit_transform`!) on `X_test_assign` to get `X_test_assign_enc`.

In [ ]:
encoder_final = OneHotEncoder(use_cat_names=True)
X_train_assign_enc = encoder_final.fit_transform(X_train_assign)  # <---  X_train_assign
X_test_assign_enc = encoder_final.transform(X_test_assign)       # <---  X_test_assign

---

## Part 4: Evaluation Metrics — Precision, Recall, and the Cost of Being Wrong

### Why Accuracy Alone Is Not Enough

Accuracy = (correct predictions) / (total predictions) seems natural, but it can be deeply misleading with imbalanced classes:

**Extreme example:** if 90% of buildings are not severely damaged, a model that always predicts "not severe" gets 90% accuracy — but it catches *zero* actually severe buildings. Perfectly useless.

In Kavrepalanchok, ~54.7% of buildings are severely damaged, so the class balance is relatively mild. But the principle still applies: **accuracy alone doesn't tell you whether you're catching the right buildings**.

### The Confusion Matrix — Four Outcomes

```
                            Predicted
                      | Not Severe | Severe |
    Actual Not Sev    |     TN     |   FP   |
    Actual Severe     |     FN     |   TP   |

TN = True Negative  : Correctly predicted not severe  ✅
TP = True Positive  : Correctly predicted severe      ✅
FP = False Positive : Predicted severe, actually not  ❌ (false alarm)
FN = False Negative : Predicted not severe, actually is severe ❌ (MISSED DANGER)
```

### The Disaster-Response Cost Asymmetry

> ⚠️ **In earthquake disaster response, false negatives are far more costly than false positives.**

**False Positive (FP):** we predict a building is severely damaged, but it's actually safe.
- Cost: an inspection team is dispatched unnecessarily
- Outcome: wasted resources, minor inconvenience to occupants temporarily displaced
- **Severity: LOW** — recoverable, no lives at risk

**False Negative (FN):** we predict a building is safe, but it's actually severely damaged.
- Cost: the building is NOT flagged for inspection
- Outcome: occupants remain in a structurally compromised building → if the building collapses in an aftershock, people are killed
- **Severity: CATASTROPHIC** — irreversible, potential loss of life

**The asymmetry:**

```
FN cost ≫ FP cost

In dollar terms:
  FP: dispatch inspection team → ~\$200 per building
  FN: building collapses with occupants → loss of life, trauma, full rebuild costs

In disaster relief terms:
  FP: flag 1,000 extra buildings for inspection → extra week of inspection work
  FN: miss 50 severely damaged buildings → potential fatalities in aftershocks
```

> ⚠️ **This asymmetry has a direct implication for model design:** we should prefer models and thresholds that maximize **Recall** (catching as many severe buildings as possible), even at the cost of some **Precision** (accepting more false alarms). A model that achieves 75% recall at 65% precision is *better for disaster response* than one that achieves 90% precision at 55% recall — even if the latter has higher accuracy.

### Precision and Recall — The Fundamental Tradeoff

**Precision** = TP / (TP + FP)
- "Of all buildings we predicted severe, how many actually are?"
- High precision = fewer false alarms, but may miss some severe buildings

**Recall** = TP / (TP + FN)
- "Of all buildings that are actually severe, how many did we correctly flag?"
- High recall = we catch most severe buildings, but more false alarms

**The tradeoff:**
```
Threshold 0.7 → Precision = 0.85, Recall = 0.60  (misses 40% of severe buildings)
Threshold 0.3 → Precision = 0.65, Recall = 0.90  (catches 90%, more false alarms)
```

Lowering the decision threshold increases recall (flag more buildings as severe) at the cost of precision. In disaster response, **a threshold below 0.5 is usually appropriate**.

### Class Imbalance Reminder

Always check class distribution before choosing metrics. If 95% of buildings were not severe:
- A "predict all not severe" model: 95% accuracy, 0% recall on severe class
- This model would be catastrophically bad for disaster response despite its high accuracy

Rule: **always report precision AND recall alongside accuracy** for disaster-response classification.

---

## 3. Modeling and Evaluation

**Code Task 4.5.3.1**: Train a `DecisionTreeClassifier` with `max_depth=10` and `random_state=42`. Name it `assignment_tree_model`. Fit it on `X_train_assign_enc` and `y_train_assign`. Print training accuracy to verify the model fitted correctly.

In [ ]:
assignment_tree_model = DecisionTreeClassifier(max_depth=10, random_state=42)   # <---  max_depth=10
assignment_tree_model.fit(X_train_assign_enc, y_train_assign)                                              # <---  X_train_assign_enc, y_train_assign

Now evaluate your model on the **test set** — the 20% of data the model has never seen.

Compute three metrics:
- **Accuracy** (`final_score_acc`): overall fraction correct — useful context but not the primary metric
- **Precision** (`final_score_prec`): of buildings predicted severe, fraction that actually are
- **Recall** (`final_score_rec`): of buildings that actually are severe, fraction we correctly flag

> 📌 **Expected ranges for Kavrepalanchok:** accuracy ~70-75%, precision ~65-80%, recall ~70-85%. If accuracy is much lower than the majority-class baseline (~55%), something is wrong. If recall is below 50%, the model is missing too many severe buildings for disaster-response use.

**Code Task 4.5.3.2**: Calculate `accuracy_score`, `precision_score`, and `recall_score` on the test set. Store in `final_score_acc`, `final_score_prec`, `final_score_rec`.

In [ ]:
y_pred_assign = assignment_tree_model.predict(X_test_assign_enc)       # <---  X_test_assign_enc

final_score_acc = accuracy_score(y_test_assign, y_pred_assign)               # <---  y_test_assign, y_pred_assign
final_score_prec = precision_score(y_test_assign, y_pred_assign)             # <---  y_test_assign, y_pred_assign
final_score_rec = recall_score(y_test_assign, y_pred_assign)                 # <---  y_test_assign, y_pred_assign

print(f"Final Accuracy: {final_score_acc:.4f}")

**Multiple Choice Question 4.5.3.1**

Which metric do you use for overall "correctness"?

1. Precision
2. Recall
3. Accuracy [x]
4. F1-score

> 📊 **Interpreting your metrics:**
> - If `final_score_acc` ≈ 54.7%, your model is essentially the majority-class baseline — it needs more depth or better features
> - If `final_score_rec` < 0.60, the model misses 40%+ of severely damaged buildings — too risky for disaster response
> - If `final_score_prec` < 0.50 but `final_score_rec` > 0.80, consider the tradeoff: high recall at the cost of many false alarms

---

## 4. Communication and Interpretation

A model's accuracy is only part of the story. **Feature importance** tells us *which features* the model relied on most — connecting the prediction to physical intuition.

**Code Task 4.5.4.1**: Extract `feature_importances_` from `assignment_tree_model`. Get feature names from `encoder_final.get_feature_names_out()` (or the equivalent for your encoder). Identify the single most important feature and store its name in `most_important_feature_name`.

In [ ]:
importances_assign = assignment_tree_model.feature_importances_
feat_names_assign = X_train_assign_enc.columns
feat_imp_assign_series = pd.Series(importances_assign, index=feat_names_assign)             # <---  importances_assign, index=feat_names_assign

most_important_feature_name = feat_imp_assign_series.idxmax()
print(f"Most important feature: {most_important_feature_name}")

**Multiple Choice Question 4.5.4.1**

Why look at feature importance?

1. To ignore features.
2. To understand characteristics associated with risk. [x]
3. To make predictions binary.
4. To keep connection open.

> 📊 **Interpreting feature importance:**
> - The top features should align with structural engineering knowledge: `foundation_type`, `roof_type`, and `age_building` typically rank highest
> - If `vdcmun_id` (municipality ID) ranks very high, it suggests strong geographic clustering of damage — the tree is essentially learning "which municipality" rather than "what building characteristics"
> - Compare your Kavrepalanchok top features against Gorkha results (Lesson 3) — are the same features important, or does this district have different primary drivers?

**Code Example 4.5.4.2**: Visualize model results — confusion matrix and feature importance chart


In [ ]:
# Create subplots for confusion matrix and feature importance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 6))

# Plot confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred_final = assignment_tree_model.predict(X_test_assign_enc)
cm = confusion_matrix(y_test_assign, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Severe', 'Severe'])
disp.plot(ax=ax1, values_format='d')
ax1.set_title('Confusion Matrix: Kavrepalanchok Model')

# Plot top 10 feature importances
top_10_features = feat_imp_assign_series.sort_values(ascending=False).head(10)
top_10_features.plot(kind='barh', ax=ax2)
ax2.set_xlabel('Importance')
ax2.set_title('Top 10 Most Important Features')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print("\nModel Summary for Kavrepalanchok:")
print(f"- Total buildings: {len(df_assignment_final)}")
print(f"- Severe damage rate: {df_assignment_final['severe_damage'].mean():.1%}")
print(f"- Test accuracy: {final_score_acc:.3f}")
print(f"- Most important feature: {most_important_feature_name}")

## 5. Summary and Final Discussion

Congratulations on completing the P4 End-to-End Assignment! You have built a complete classification pipeline from raw database to model evaluation — the core workflow of data science practice.

### What You Accomplished

| Step | What You Did | Key Decision |
|------|-------------|-------------|
| **Data Acquisition** | Four-table SQL JOIN for Kavrepalanchok | Used `district_id = 3`; applied INNER JOIN to get complete records |
| **Target creation** | `damage_grade` → binary `severe_damage` | Grades 4-5 = 1 (must evacuate); 1-3 = 0 |
| **Wrangling** | Dropped leaky columns, handled missing values | Excluded post-earthquake measurements |
| **Splitting** | 80/20 train/test with `random_state=42` | Test set sealed until final evaluation |
| **Encoding** | `OneHotEncoder(use_cat_names=True)` | Fit on training only; transformed test |
| **Modeling** | `DecisionTreeClassifier(max_depth=10)` | Depth chosen from Lesson 3 validation curve |
| **Evaluation** | Accuracy, Precision, Recall on test set | Three metrics to capture different aspects of correctness |
| **Interpretation** | Feature importances extracted and visualized | Connected model output to physical meaning |

### Key Results

- **Accuracy:** your model should achieve approximately 70-75% accuracy on the Kavrepalanchok test set — notably better than the majority-class baseline (~55% for this district)
- **Most important feature:** `foundation_type` and `age_building` typically rank highest, consistent with Lessons 2 and 3 on Gorkha data — structural features dominate
- **District comparison:** Kavrepalanchok has a different class balance (~55% severe) than Gorkha (~64% severe), reflecting different building stock composition or different shaking intensity

### The Disaster-Response Cost Asymmetry — Revisited

The metric discussion in Part 4 revealed a fundamental design choice: in earthquake damage prediction, **recall matters more than precision**.

- A model with 80% recall catches 80% of all severely damaged buildings — flagging them for inspection
- The 20% it misses may contain buildings where occupants remain at risk in aftershocks
- A false alarm (flagging a safe building) costs money and inconvenience — but not lives

**When you look at your results, ask:** is your recall above 70%? If not, consider what you would change — lower the decision threshold, use `class_weight='balanced'`, or tune `max_depth` to a different value.

### Equity Reflection

Kavrepalanchok, like Gorkha, has communities where certain caste groups are concentrated in older buildings with weaker foundations. The feature importance ranking you computed tells part of this story:

- If `foundation_type` is the dominant feature, it means building quality — not demographics — is the primary predictor. Reconstruction programs should prioritize foundation retrofitting.
- If `caste_household` features appear in the top 10, it suggests demographic factors have residual predictive power beyond structural features alone.

**In either case:** the downstream equity question is the same — *are reconstruction resources reaching the communities most in need?*

### Discussion Questions

1. Your model achieved approximately X% accuracy on Kavrepalanchok. How does this compare to the majority-class baseline? What does the difference tell you?
2. The top feature in your importance chart is typically `foundation_type`. Does this make physical sense? What property of RC foundations makes them protective?
3. If you wanted to maximize recall (catch as many severely damaged buildings as possible), what is one concrete change you could make to the current pipeline? What would be the downside?
4. You used `max_depth=10` without tuning it specifically for Kavrepalanchok. How would you determine the optimal max_depth for this district? What tool would you use?
5. Suppose a relief agency will use your model to decide which buildings to inspect first. What threshold would you recommend — 0.5 (default) or lower? Justify your answer using the cost-asymmetry framework from Part 4.
6. Compare your Kavrepalanchok results to Gorkha (Lessons 2-4). Are the same features most important? Is the model accuracy similar? What might explain any differences?

### What This Assignment Demonstrated

You now have the complete P4 toolkit:

| Skill | Where Used |
|-------|-----------|
| SQL JOINs with DuckDB | L1 + this assignment |
| Binary classification framing | L2, L3, L4, this assignment |
| Logistic Regression pipeline | L2 |
| Decision Tree + validation curve | L3 |
| Demographic analysis + equity framing | L4 |
| End-to-end pipeline | This assignment |

> ➡️ **In Unit 2,** you will extend these skills to ensemble methods (Random Forests, Gradient Boosting), cross-validation, and more sophisticated feature engineering. The classification foundations you built in P4 are the building blocks for everything that follows.
